## Imports and config

In [ ]:
import ast
import csv
import os
from pprint import pprint

QA_DIR_PATH = "../qa_datasets"
RESULTS_DIR_PATH = "../results"

TEMPLATE_ID_TO_HOPS = {
    0: 1,
    1: 1,
    2: 2,
    3: 2,
    4: 3,
    5: 3,
}


def get_empty_metrics():
    return {
        "Hit@1": [],
        "Hit@5": [],
        "Recall@20": [],
        "MRR": [],
    }


def compute_averages(metrics_by_retriever):
    averages = {}
    for retriever_name, metrics in metrics_by_retriever.items():
        averages[retriever_name] = {
            key: round(100 * sum(values) / len(values), 2) if values else 0.0
            for key, values in metrics.items()
        }
    return averages

In [ ]:
metrics_per_graph = {}

eval_graphs = os.listdir(RESULTS_DIR_PATH)
for eval_graph in eval_graphs:
    metrics_per_graph[eval_graph] = {}

    eval_graph_path = os.path.join(RESULTS_DIR_PATH, eval_graph)
    eval_datasets = os.listdir(eval_graph_path)
    for eval_dataset in eval_datasets:
        qa_file_path = os.path.join(QA_DIR_PATH, f"{eval_dataset}.csv")
        if not os.path.exists(qa_file_path):
            print(f"{eval_dataset} does not exist in QA folder, skipping...")
            continue

        eval_dataset_path = os.path.join(eval_graph_path, eval_dataset)

        retrievers_data = {}
        retriever_metrics = {}
        for retriever_file in os.listdir(eval_dataset_path):
            if not retriever_file.endswith(".csv"):
                continue

            retriever_name = os.path.splitext(retriever_file)[0]

            retrievers_data[retriever_name] = {}
            with open(
                os.path.join(eval_dataset_path, retriever_file),
                mode="r",
                encoding="utf-8",
            ) as f:
                reader = csv.reader(f)
                next(reader)

                for row in reader:
                    q_id = str(row[0]).strip()
                    retrievers_data[retriever_name][q_id] = ast.literal_eval(row[1])

            retriever_metrics[retriever_name] = get_empty_metrics()

        hop_metrics = {}

        with open(qa_file_path, mode="r", encoding="utf-8") as dataset_file:
            dataset_reader = csv.DictReader(dataset_file)
            has_template_col = bool("template_id" in dataset_reader.fieldnames)

            for row in dataset_reader:
                query_id = row["id"]
                answer_ids = set(ast.literal_eval(row["answer_ids"]))

                hop = None
                if has_template_col:
                    template_id = int(row["template_id"])
                    hop = TEMPLATE_ID_TO_HOPS[template_id]
                    if hop not in hop_metrics:
                        hop_metrics[hop] = {
                            retriever_name: get_empty_metrics()
                            for retriever_name in retrievers_data
                        }

                for retriever_name, retr_ids_per_query in retrievers_data.items():
                    retrieved_ids = retr_ids_per_query[query_id]

                    # Hit@1
                    hit1 = (
                        1
                        if len(retrieved_ids) > 0 and retrieved_ids[0] in answer_ids
                        else 0
                    )

                    # Hit@5
                    hit5 = (
                        1
                        if any(retr_id in answer_ids for retr_id in retrieved_ids[:5])
                        else 0
                    )

                    # Recall@20
                    recall20 = len(set(retrieved_ids[:20]) & answer_ids) / len(
                        answer_ids
                    )

                    # MRR
                    mrr = 0.0
                    for rank, retr_id in enumerate(retrieved_ids, start=1):
                        if retr_id in answer_ids:
                            mrr = 1.0 / rank
                            break

                    targets = [retriever_metrics[retriever_name]]
                    if hop is not None:
                        targets.append(hop_metrics[hop][retriever_name])

                    for target_metrics in targets:
                        target_metrics["Hit@1"].append(hit1)
                        target_metrics["Hit@5"].append(hit5)
                        target_metrics["Recall@20"].append(recall20)
                        target_metrics["MRR"].append(mrr)

        metrics_per_graph[eval_graph][eval_dataset] = compute_averages(
            retriever_metrics
        )
        for hop in sorted(hop_metrics):
            hop_dataset_name = f"{eval_dataset} ({hop}-Hop)"
            metrics_per_graph[eval_graph][hop_dataset_name] = compute_averages(
                hop_metrics[hop]
            )

pprint(metrics_per_graph["stark-amazon"], sort_dicts=False)